# F1TENTH paper table: Atlas-DeePC versus Contextual-DPC

Seven measurement-noise levels; one Atlas configuration per level and Contextual-DPC with 30/60/90 columns. Each row uses 20 fresh paired online seeds (531–550). Atlas K and rank d are selected on separate development seeds and frozen before these evaluations. Both methods use the same noisy offline dataset at each level.

NSR means the **standard-deviation ratio** sigma: y_meas,j = y_true,j + sigma s_j epsilon_j. Noise is added offline before clustering and online with the same channel standard deviations. The levels are 0%, 0.1%, 0.25%, 0.5%, 1%, 2.5% and 5%.

[Protocol](../../../docs/f1tenth/README.md). Run All reads the saved table by default. The saved table uses the historical commanded-input convention. Enable the flag below to evaluate the same 560 configurations/seeds with the new applied-input convention; these are new results, not an exact reproduction of the saved table.

In [1]:
from pathlib import Path
import os, sys, json
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[name]='1'
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/atlas_deepc').is_dir())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
import pandas as pd
from IPython.display import display, Markdown
from atlas_deepc.f1tenth.paths import RESULTS_DIR
RUN_ALL_EXPERIMENTS=False
PAPER_ROOT=RESULTS_DIR/'paper_nsr_20261027'
RERUN_ROOT=RESULTS_DIR/'paper_applied_input_rerun'
if RUN_ALL_EXPERIMENTS:
    from run_f1tenth_paper import run_paper
    run_paper(RERUN_ROOT,selection=PAPER_ROOT/'selection')
RESULT_ROOT=RERUN_ROOT if RUN_ALL_EXPERIMENTS else PAPER_ROOT
status=json.loads((RESULT_ROOT/'status.json').read_text())
print(status)
summary=json.loads((RESULT_ROOT/'report.json').read_text()) if (RESULT_ROOT/'report.json').exists() else []


{'status': 'complete', 'levels': 7, 'rollouts': 560, 'configurations_frozen_before_evaluation': True}


## Paper comparison

RMSE is the median cross-track RMSE over collision-free completed laps. A dash means no lap succeeded. Geometric completion alone does not imply collision-free driving. Controller timing includes selection, reference construction, assembly, setup and solve, excluding the first five calls; it includes failed prefixes.

The objectives have different additional terms. Atlas uses no g regularization or coefficient bounds. Its K,d varies by noise level; Contextual uses fixed regularization settings. All other task, noise and reference settings are shared.

In [2]:
rows=[]
for s in summary:
    rows.append({'NSR (%)':100*s['sigma'],'Method':s['label'],
        'Collision-free laps':f"{s['collision_free_laps']}/{s['evaluated']}",
        'Geometric laps':f"{s['geometric_laps']}/{s['evaluated']}",
        'Median RMSE (cm)':s['median_rmse_cm'],
        'Median controller time (ms)':s['timing']['median_ms'] if s['timing'] else None})
if rows:
    display(pd.DataFrame(rows).round(3).fillna('—'))
    display(Markdown(f"[LaTeX table]({RESULT_ROOT/'table.tex'}) · [Report]({RESULT_ROOT/'report.md'})"))
else:
    print('Configuration search is in progress; fresh paper results will appear after selection is locked.')


,NSR (%),Method,Collision-free laps,Geometric laps,Median RMSE (cm),Median controller time (ms)
0,0.00,"Atlas-DeePC (K=2, d=42)",20/20,20/20,1.754,2.448
1,0.00,Contextual-DPC (30 columns),20/20,20/20,5.201,3.872
2,0.00,Contextual-DPC (60 columns),20/20,20/20,2.611,9.765
3,0.00,Contextual-DPC (90 columns),20/20,20/20,1.943,13.700
4,0.10,"Atlas-DeePC (K=12, d=40)",20/20,20/20,1.745,2.427
5,0.10,Contextual-DPC (30 columns),20/20,20/20,4.874,3.884
6,0.10,Contextual-DPC (60 columns),20/20,20/20,2.675,9.957
7,0.10,Contextual-DPC (90 columns),20/20,20/20,2.063,13.764
8,0.25,"Atlas-DeePC (K=13, d=42)",20/20,20/20,3.413,2.711
9,0.25,Contextual-DPC (30 columns),20/20,20/20,6.099,3.897


[LaTeX table](/Users/kumiko/Projects/atlas_deepc_reacher/experiments/f1tenth/paper_nsr_20261027/table.tex) · [Report](/Users/kumiko/Projects/atlas_deepc_reacher/experiments/f1tenth/paper_nsr_20261027/report.md)

## Selection diagnostics

At sufficiently large ranks, chart past images may span the entire aligned 27-dimensional history space. Projection residuals then tie and the prescribed smallest-ID rule selects a fixed chart. Successful tracking in this case does not demonstrate an adaptive switching benefit. There is one shared offline-noise realization per level; the fresh seeds vary online noise and initialization.

In [3]:
diagnostic_file=RESULT_ROOT/'selection_diagnostics.json'
if diagnostic_file.exists():
    diagnostic=json.loads(diagnostic_file.read_text())
    display(pd.DataFrame([{'NSR (%)':100*s['sigma'],'K':s['config']['clusters'],'d':s['config']['rank'],
        'Past ranks':s['past_ranks'],'Chart call counts':s['chart_counts'],
        'Tie fraction':s['tie_fraction'],'Switch fraction':s['switch_fraction']} for s in diagnostic]))
else:
    print('Diagnostics will be available with the paper evaluations.')


,NSR (%),K,d,Past ranks,Chart call counts,Tie fraction,Switch fraction
0,0.00,2,42,"[23, 27]","[0, 4400]",0.0,0.0
1,0.10,12,40,"[27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27]","[4420, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0
2,0.25,13,42,"[27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 2...","[4406, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0
3,0.50,14,60,"[27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 2...","[4405, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0
4,1.00,10,44,"[27, 27, 27, 27, 27, 27, 27, 27, 27, 27]","[4414, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0
5,2.50,3,68,"[27, 27, 27]","[4056, 0, 0]",1.0,0.0
6,5.00,8,72,"[27, 27, 27, 27, 27, 27, 27, 27]","[2472, 0, 0, 0, 0, 0, 0, 0]",1.0,0.0
